**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_motif_richard
analysis_variant_motif_richard_arc251231
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_pilot
batches_top
motifdelta_top_jaspar2024
motifdelta_top_jvierstra_v2.1beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.pmap.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
motifscan_top_jaspar2024
motifscan_top_jvierstra_v2.1beta
variant_closed_gof_bluestarr.flankL35R70.ref.fa
variant_closed_gof_bluestarr.tsv


In [4]:
ls ${FD_RES}/analysis_variant_motif_richard/batches_top

variant_closed_gof_bluestarr.flankL35R70.top01k.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.top10k.ref.fa.gz


## Jaspar2024

### Prepare

In [5]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_jaspar2024
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_jaspar2024

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

In [6]:
ls ${FD_MSCAN}

variant_closed_gof_bluestarr.flankL35R70.top01k.npz
variant_closed_gof_bluestarr.flankL35R70.top10k.npz


### Execute

In [7]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_02_delta.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A majoroslab
    -p igvf,common
    --exclude=dcc-comp-10
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set I/O
    TXT_JOB=motifdelta.jaspar.${TXT_TAG}
    FP_INP=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.npz
    FP_OUT_PREFIX=${FD_DELTA}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}
    FP_LOG=${FD_LOG}/run.motifdelta.jaspar.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifdelta.jaspar.batch.${TXT_TAG}.%j.txt

    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=50G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_TBIND}" "${FP_OUT_PREFIX}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 42206504
Submitted top10k: 42206505


## Review

In [8]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42206504.ba+                          batch  COMPLETED   00:00:13  00:04.702   5070080K 

===== ElapsedRaw =====
ElapsedRaw = 13 sec (0.22 min)

===== MaxRSS =====
MaxRSS = 4.84 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42206505.ba+                          batch  COMPLETED   00:00:54  00:30.347  48937012K 

===== ElapsedRaw =====
ElapsedRaw = 54 sec (0.90 min)

===== MaxRSS =====
MaxRSS = 46.67 GiB



In [9]:
#cat ${FD_LOG}/run.motifdelta.jaspar.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifdelta.jaspar.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-comp-08
Slurm Array Index:  NA
Time Stamp:         01-19-26+14:06:23
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading scan results: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.top01k.npz
N=1000, M=879, P=74, strands=2
Load and check in 3.22 seconds

Loading motif Tbind: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
Loaded Tbind for 879 motifs.
Load and check in 0.01 seconds

Computing reduced best gain/loss per variant × motif...
Compute reduced delta + best events in 1.14 seconds

Extracting gain/loss events (one gain + one loss per variant × motif)...
Summarizing motif-level gain/loss...
Summarizing variant-level gain/loss...
Events + summaries in 0.10 seconds

Saving reduced delta NPZ to /hpc/group/igvf/kk319/repo/Proj_IGVF

In [11]:
#cat ${FD_LOG}/run.motifdelta.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifdelta.jaspar.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-comp-08
Slurm Array Index:  NA
Time Stamp:         01-19-26+14:06:23
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading scan results: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jaspar2024/variant_closed_gof_bluestarr.flankL35R70.top10k.npz
N=10000, M=879, P=74, strands=2
Load and check in 30.04 seconds

Loading motif Tbind: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
Loaded Tbind for 879 motifs.
Load and check in 0.00 seconds

Computing reduced best gain/loss per variant × motif...
Compute reduced delta + best events in 11.38 seconds

Extracting gain/loss events (one gain + one loss per variant × motif)...
Summarizing motif-level gain/loss...
Summarizing variant-level gain/loss...
Events + summaries in 0.87 seconds

Saving reduced delta NPZ to /hpc/group/igvf/kk319/repo/Proj_I

## Non-redundant motifs

### Prepare

In [12]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_top
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_top_jvierstra_v2.1beta

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

In [25]:
ls -1 ${FD_MSCAN}

variant_closed_gof_bluestarr.flankL35R70.top01k.npz
variant_closed_gof_bluestarr.flankL35R70.top10k.npz


### Execute

In [27]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_02_delta.sh

### set slurm opt
NUM_CPU=2
#NUM_MEM=40G
#NUM_MEM=20G

LST_OPTS=(
    -A majoroslab
    -p igvf,common
    --exclude=dcc-comp-10
    --cpus-per-task="${NUM_CPU}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
LST_TAGS=(top01k top10k)

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set I/O
    TXT_JOB=motifdelta.jvierstra.${TXT_TAG}
    FP_INP=${FD_MSCAN}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}.npz
    FP_OUT_PREFIX=${FD_DELTA}/variant_closed_gof_bluestarr.flankL35R70.${TXT_TAG}
    FP_LOG=${FD_LOG}/run.motifdelta.jvierstra.batch.${TXT_TAG}.txt
    #FP_LOG=${FD_LOG}/run.motifdelta.jaspar.batch.${TXT_TAG}.%j.txt

    ### set memory
    if [[ "${TXT_TAG}" == "top01k" ]]; then
        NUM_MEM=8G
    else
        NUM_MEM=50G
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --mem="${NUM_MEM}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_TBIND}" "${FP_OUT_PREFIX}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 42206925
Submitted top10k: 42206926


## Review

In [29]:
for JOBID in "${LST_JOBS[@]}"; do
    sacct_summary.sh "${JOBID}"
    echo
done

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42206925.ba+                          batch  COMPLETED   00:00:12  00:04.184   2780328K 

===== ElapsedRaw =====
ElapsedRaw = 12 sec (0.20 min)

===== MaxRSS =====
MaxRSS = 2.65 GiB

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
42206926.ba+                          batch  COMPLETED   00:03:40  03:09.945  37949820K 

===== ElapsedRaw =====
ElapsedRaw = 220 sec (3.67 min)

===== MaxRSS =====
MaxRSS = 36.19 GiB



In [30]:
#cat ${FD_LOG}/run.motifdelta.jvierstra.batch.${LST_TAGS[0]}.${LST_JOBS[0]}.txt
cat ${FD_LOG}/run.motifdelta.jvierstra.batch.${LST_TAGS[0]}.txt

Hostname:           dcc-core-02
Slurm Array Index:  NA
Time Stamp:         01-19-26+14:18:44
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading scan results: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.top01k.npz
N=1000, M=637, P=79, strands=2
Load and check in 1.16 seconds

Loading motif Tbind: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
Loaded Tbind for 637 motifs.
Load and check in 0.00 seconds

Computing reduced best gain/loss per variant × motif...
Compute reduced delta + best events in 0.96 seconds

Extracting gain/loss events (one gain + one loss per variant × motif)...
Summarizing motif-level gain/loss...
Summarizing variant-level gain/loss...
Events + summaries in 0.12 seconds

Saving reduced delta NPZ to /hpc/group/igvf/kk319/repo/Proj_IGVF_

In [31]:
#cat ${FD_LOG}/run.motifdelta.jaspar.batch.${LST_TAGS[1]}.${LST_JOBS[1]}.txt
cat ${FD_LOG}/run.motifdelta.jvierstra.batch.${LST_TAGS[1]}.txt

Hostname:           dcc-core-46
Slurm Array Index:  NA
Time Stamp:         01-19-26+14:18:47
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading scan results: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_top_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.top10k.npz
N=10000, M=637, P=79, strands=2
Load and check in 32.93 seconds

Loading motif Tbind: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
Loaded Tbind for 637 motifs.
Load and check in 0.04 seconds

Computing reduced best gain/loss per variant × motif...
Compute reduced delta + best events in 167.79 seconds

Extracting gain/loss events (one gain + one loss per variant × motif)...
Summarizing motif-level gain/loss...
Summarizing variant-level gain/loss...
Events + summaries in 1.20 seconds

Saving reduced delta NPZ to /hpc/group/igvf/kk319/repo/Proj_I

In [32]:
sbatch -p common --nodelist=dcc-comp-02 --wrap="hostname; date; python -c 'print(\"hello\")'"

Submitted batch job 42207080


In [26]:
sinfo -o "%P %D %t %N" | head -n 50

PARTITION NODES STATE NODELIST
common* 5 comp dcc-comp-03,dcc-core-[04,13,44,51]
common* 3 drng dcc-core-[42,49,55]
common* 31 mix dcc-comp-[01-02,04,10],dcc-core-[02,08,12,15-16,24,28-31,33-36,38-41,43,45-48,50,52-54]
common* 10 alloc dcc-comp-[05-09],dcc-core-[03,05-06,14,37]
common* 15 idle dcc-core-[07,09-11,17-23,25-27,32]
gpu-common 1 comp dcc-core-gpu-09
gpu-common 9 mix- dcc-core-gpu-[10-11,27-29,32,40-42]
gpu-common 3 mix dcc-core-gpu-34,dcc-core-gpu-ferc-g-r32-1,dcc-core-gpu-ferc-s-p15-20
gpu-common 15 alloc dcc-core-gpu-[16-17,26,30-31,33,35-39,43-46]
gpu-common 4 idle dcc-core-gpu-[05-08]
scavenger 5 comp dcc-courses-31,dcc-liulab-02,dcc-mism-ferc-u-ab25-4-3,dcc-physics-01,dcc-tunglab-01
scavenger 1 drng dcc-tunglab-02
scavenger 2 drain dcc-barthellab-01,dcc-dhvi-md-15
scavenger 69 mix dcc-adrc-01,dcc-barthellab-[02-03],dcc-cagpm-02,dcc-caperlab-01,dcc-chsi-[13-15,17-22],dcc-comp-[01-02,10,12],dcc-cosmology-[09-10],dcc-courses-[01-17,21-30,39,45,47-49],dcc-delairelab-01,dcc

In [28]:
sacct -u $USER -S now-2days \
  --format=JobID,JobName%30,State,ExitCode,Reason,Elapsed,NodeList%30 \
  -P | egrep "CANCELLED|FAILED|TIMEOUT|PREEMPTED"

42173424|bash|CANCELLED by 1155335|0:0|None|00:08:05|dcc-allenlab-01
42173424.0|bash|CANCELLED|0:9||00:08:35|dcc-allenlab-01
42173425|bash|CANCELLED by 1155335|0:0|None|00:07:58|dcc-allenlab-01
42173425.0|bash|CANCELLED|0:9||00:08:28|dcc-allenlab-01
42173448|bash|CANCELLED by 1155335|0:0|None|00:00:00|None assigned
42173451|bash|CANCELLED by 1155335|0:0|None|00:00:00|None assigned
42174145|bash|CANCELLED by 1155335|0:0|None|00:00:00|None assigned
42176508|motifdelta.jaspar.top01k|CANCELLED by 1155335|0:0|None|00:18:35|dcc-core-04
42176508.batch|batch|CANCELLED|0:0||00:18:35|dcc-core-04
42204374|meme2lods_jaspar2024|CANCELLED by 1155335|0:0|None|00:00:00|None assigned
42204465|run_meme2lods_jvierstra|CANCELLED by 1155335|0:0|None|00:00:00|None assigned
42205794|motifdelta.jaspar.top10k|CANCELLED by 1155335|0:0|None|00:02:19|dcc-comp-02
42205794.batch|batch|CANCELLED|0:15||00:02:20|dcc-comp-02
42206119|motifscan.jaspar.top01k|FAILED|0:53|None|00:00:11|dcc-comp-10
42206119.batch|batch|CAN

In [33]:
sinfo -p common -N -h -o "%N %T %E" | egrep "^dcc-core-"

dcc-core-02 mixed none
dcc-core-03 allocated none
dcc-core-04 completing none
dcc-core-05 allocated none
dcc-core-06 allocated none
dcc-core-07 idle none
dcc-core-08 mixed none
dcc-core-09 idle none
dcc-core-10 idle none
dcc-core-11 idle none
dcc-core-12 mixed none
dcc-core-13 completing none
dcc-core-14 allocated none
dcc-core-15 mixed none
dcc-core-16 mixed none
dcc-core-17 idle none
dcc-core-18 idle none
dcc-core-19 idle none
dcc-core-20 idle none
dcc-core-21 idle none
dcc-core-22 idle none
dcc-core-23 idle none
dcc-core-24 mixed none
dcc-core-25 idle none
dcc-core-26 idle none
dcc-core-27 idle none
dcc-core-28 mixed none
dcc-core-29 mixed none
dcc-core-30 mixed none
dcc-core-31 mixed none
dcc-core-32 idle none
dcc-core-33 mixed none
dcc-core-34 mixed none
dcc-core-35 mixed none
dcc-core-36 mixed none
dcc-core-37 mixed none
dcc-core-38 mixed none
dcc-core-39 mixed none
dcc-core-40 mixed none
dcc-core-41 mixed none
dcc-core-42 draining Kill task failed
dcc-core-43 mixed none
dcc-core

In [34]:
sinfo -p common -N -h -o "%N %T %E" | egrep "^dcc-comp-"

dcc-comp-01 allocated none
dcc-comp-02 allocated none
dcc-comp-03 completing none
dcc-comp-04 allocated none
dcc-comp-05 allocated none
dcc-comp-06 mixed none
dcc-comp-07 allocated none
dcc-comp-08 mixed none
dcc-comp-09 allocated none
dcc-comp-10 mixed none


In [35]:
sacctmgr show assoc user=$USER format=User,Account,Partition,QOS -P

User|Account|Partition|QOS
kk319|biostat||normal
kk319|chsi||normal
kk319|h200ea||normal
kk319|igvf||normal
kk319|majoroslab||normal
kk319|reddylab||normal


In [18]:
squeue -j 42176508 -o "%.18i %.9P %.8T %.10M %.6D %R"

             JOBID PARTITION    STATE       TIME  NODES NODELIST(REASON)
          42176508    common  RUNNING       9:32      1 dcc-core-04


In [19]:
scontrol show job 42176508 | egrep -i "StdOut|StdErr|WorkDir|Command|BatchHost|NodeList"

   ReqNodeList=(null) ExcNodeList=(null)
   NodeList=dcc-core-04
   BatchHost=dcc-core-04
   Command=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts/run_motifdelta_02_delta.sh
   WorkDir=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
   StdErr=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log/run.motifdelta.jaspar.batch.top01k.42176508.txt
   StdOut=/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log/run.motifdelta.jaspar.batch.top01k.42176508.txt


In [20]:
sstat -j 42176508.batch --format=JobID,AveCPU,AveRSS,MaxRSS,MaxVMSize

JobID            AveCPU     AveRSS     MaxRSS  MaxVMSize 
------------ ---------- ---------- ---------- ---------- 
42176508.ba+ 213503982+                                  


In [21]:
ls -l /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log/run.motifdelta.jaspar.batch.top01k.42176508.txt


ls: cannot access '/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log/run.motifdelta.jaspar.batch.top01k.42176508.txt': No such file or directory


: 2

In [22]:
sacct -j 42176508 --format=JobID,State,Elapsed,CPUTime,MaxRSS,AllocCPUS,NodeList

JobID             State    Elapsed    CPUTime     MaxRSS  AllocCPUS        NodeList 
------------ ---------- ---------- ---------- ---------- ---------- --------------- 
42176508        RUNNING   00:14:16   00:28:32                     2     dcc-core-04 
42176508.ba+    RUNNING   00:14:16   00:28:32                     2     dcc-core-04 
42176508.ex+    RUNNING   00:14:16   00:28:32                     2     dcc-core-04 


In [23]:
sstat -j 42176508.ba+ --format=JobID,AveCPU,AveRSS,MaxRSS,MaxVMSize

sstat: fatal: Bad step specified: 42176508


: 1

In [24]:
ssh dcc-core-04 "uptime; free -g; vmstat 1 5"

The authenticity of host 'dcc-core-04 (10.183.19.89)' can't be established.
ED25519 key fingerprint is SHA256:H6MC2Mr6OyPT31+RiJaHA6cXwq1Bbl+UNQyz/hZmQgA.
This key is not known by any other names
Are you sure you want to continue connecting (yes/no/[fingerprint])? 



In [25]:
scancel 42176508